## 1. Load Data

In [1]:
import pandas as pd

# Load the full Bank Marketing dataset
df = pd.read_csv("data/bank-additional-full.csv", sep=";")

# Preview the first five rows
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Basic Data Inspection

In [179]:
# Check dataset dimensions
print("Shape:", df.shape)

# Display all column names
print("\nColumns:")
print(df.columns.tolist())

Shape: (41188, 21)

Columns:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


In [180]:
# Inspect data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [181]:
# Check target class counts
print("Target counts:")
print(df["y"].value_counts())

# Check target class proportions
print("\nTarget proportions:")
print(df["y"].value_counts(normalize=True))

Target counts:
y
no     36548
yes     4640
Name: count, dtype: int64

Target proportions:
y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


## 3. Data Quality Analysis

In [182]:
# Count standard missing values in each column
missing_counts = df.isnull().sum()

missing_counts

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [183]:
# Count "unknown" categorical values in each column
unknown_counts = (df == "unknown").sum()

# Show only columns containing unknown values
unknown_counts[unknown_counts > 0]

job           330
marital        80
education    1731
default      8597
housing       990
loan          990
dtype: int64

In [184]:
# Calculate percentage of unknown values
unknown_percent = unknown_counts[unknown_counts > 0] / len(df) * 100

unknown_percent.round(2)

job           0.80
marital       0.19
education     4.20
default      20.87
housing       2.40
loan          2.40
dtype: float64

In [185]:
# Summarise class distribution
class_distribution = pd.DataFrame({
    "count": df["y"].value_counts(),
    "percentage": df["y"].value_counts(normalize=True) * 100
})

class_distribution.round(2)

,count,percentage
y,,
no,36548,88.73
yes,4640,11.27


In [186]:
# Count exact duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 12


In [187]:
# Distribution of previous campaign contacts
df["previous"].value_counts().sort_index()

previous
0    35563
1     4561
2      754
3      216
4       70
5       18
6        5
7        1
Name: count, dtype: int64

In [188]:
# Distribution of previous campaign outcomes
df["poutcome"].value_counts()

poutcome
nonexistent    35563
failure         4252
success         1373
Name: count, dtype: int64

In [189]:
# Compare previous contact counts with previous campaign outcomes
pd.crosstab(df["previous"], df["poutcome"])

poutcome,failure,nonexistent,success
previous,,,
0,0,35563,0
1,3696,0,865
2,434,0,320
3,88,0,128
4,30,0,40
5,3,0,15
6,1,0,4
7,0,0,1


In [190]:
# Inspect pdays values
df["pdays"].value_counts().sort_index()

pdays
0         15
1         26
2         61
3        439
4        118
5         46
6        412
7         60
8         18
9         64
10        52
11        28
12        58
13        36
14        20
15        24
16        11
17         8
18         7
19         3
20         1
21         2
22         3
25         1
26         1
27         1
999    39673
Name: count, dtype: int64

In [191]:
# Count records using the special pdays value 999
pdays_999_count = (df["pdays"] == 999).sum()

print("Number of pdays = 999:", pdays_999_count)
print("Percentage:", round(pdays_999_count / len(df) * 100, 2), "%")

Number of pdays = 999: 39673
Percentage: 96.32 %


In [192]:
# Previous contacts exist, but pdays is still 999
previous_with_999 = (
    (df["previous"] > 0) &
    (df["pdays"] == 999)
).sum()

print("Previous > 0 and pdays = 999:", previous_with_999)

Previous > 0 and pdays = 999: 4110


In [193]:
# No previous contacts, but pdays has a recorded value
no_previous_with_pdays = (
    (df["previous"] == 0) &
    (df["pdays"] != 999)
).sum()

print("Previous = 0 and pdays != 999:", no_previous_with_pdays)

Previous = 0 and pdays != 999: 0


In [194]:
# Compare previous campaign outcome with current subscription outcome
poutcome_target = pd.crosstab(
    df["poutcome"],
    df["y"],
    normalize="index"
)

poutcome_target

y,no,yes
poutcome,,
failure,0.857714,0.142286
nonexistent,0.911678,0.088322
success,0.348871,0.651129


## 4. Remove Duplicate Rows

In [195]:
# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Confirm the updated dataset size
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (41176, 21)


In [196]:
# Confirm no duplicate rows remain
print("Remaining duplicates:", df.duplicated().sum())

Remaining duplicates: 0


## 5 Define RQ1 Feature Set

RQ1 uses a broader feature set to compare the predictive performance of different machine learning models.

The `duration` feature is excluded because it is only known after the call has ended. The original `pdays` variable is replaced by the engineered `pdays_recorded` and `pdays_clean` features.

In [197]:
# Mark whether pdays contains an actual day value
df["pdays_recorded"] = (df["pdays"] != 999).astype(int)

# Replace the sentinel value 999 with 0
df["pdays_clean"] = df["pdays"].where(df["pdays"] != 999, 0)

In [198]:
# Full feature set for RQ1 model comparison
rq1_features = [
    # Demographic and financial
    "age",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",

    # Current campaign
    "contact",
    "month",
    "day_of_week",
    "campaign",

    # Previous campaign history
    "previous",
    "poutcome",
    "pdays_recorded",
    "pdays_clean",

    # Socio-economic context
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
]

# Create RQ1 feature matrix
X_rq1 = df[rq1_features].copy()

print("RQ1 feature count:", len(rq1_features))
print("RQ1 shape:", X_rq1.shape)

RQ1 feature count: 20
RQ1 shape: (41176, 20)


## 6. Define RQ2 Feature Groups

To address RQ2, two feature sets are defined:

- **Set A:** Customer demographic and financial information only.
- **Set B:** Set A plus previous campaign history.

The comparison between these two sets will be used to measure the additional predictive value of previous campaign history.

In [199]:
# Demographic and financial features
base_features = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan"
]

print("Base features:")
print(base_features)

Base features:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']


In [200]:
# Previous campaign history features
history_features = [
    "previous",
    "poutcome",
    "pdays_recorded",
    "pdays_clean"
]

print("History features:")
print(history_features)

History features:
['previous', 'poutcome', 'pdays_recorded', 'pdays_clean']


In [201]:
# Set A: demographic + financial information
features_without_history = base_features

# Set B: base features + previous campaign history
features_with_history = base_features + history_features

print("Set A - Without history:")
print(features_without_history)

print("\nSet B - With history:")
print(features_with_history)

Set A - Without history:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

Set B - With history:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'previous', 'poutcome', 'pdays_recorded', 'pdays_clean']


In [202]:
# Create the two RQ2 feature matrices
X_without_history = df[features_without_history].copy()
X_with_history = df[features_with_history].copy()

# Encode target: no = 0, yes = 1
y = df["y"].map({"no": 0, "yes": 1})

print("Set A shape:", X_without_history.shape)
print("Set B shape:", X_with_history.shape)
print("Target shape:", y.shape)

Set A shape: (41176, 7)
Set B shape: (41176, 11)
Target shape: (41176,)


In [203]:
# Check target encoding
y.value_counts()

y
0    36537
1     4639
Name: count, dtype: int64

In [204]:
# Preview both feature sets
print("Set A:")
display(X_without_history.head())

print("Set B:")
display(X_with_history.head())

Set A:


,age,job,marital,education,default,housing,loan
0,56,housemaid,married,basic.4y,no,no,no
1,57,services,married,high.school,unknown,no,no
2,37,services,married,high.school,no,yes,no
3,40,admin.,married,basic.6y,no,no,no
4,56,services,married,high.school,no,no,yes


Set B:


,age,job,marital,education,default,housing,loan,previous,poutcome,pdays_recorded,pdays_clean
0,56,housemaid,married,basic.4y,no,no,no,0,nonexistent,0,0
1,57,services,married,high.school,unknown,no,no,0,nonexistent,0,0
2,37,services,married,high.school,no,yes,no,0,nonexistent,0,0
3,40,admin.,married,basic.6y,no,no,no,0,nonexistent,0,0
4,56,services,married,high.school,no,no,yes,0,nonexistent,0,0


## 7. Train/Test Split

The data is split into 80% training and 20% test sets using stratified sampling to preserve the class distribution of the target variable.

The same train/test observations are used for RQ1 and both RQ2 feature sets to ensure consistent and fair model comparisons.

In [205]:
from sklearn.model_selection import train_test_split

# Split row indices once so both feature sets use identical samples
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(train_idx))
print("Test samples:", len(test_idx))

Training samples: 32940
Test samples: 8236


In [206]:
# Set A: without previous campaign history
X_train_without = X_without_history.loc[train_idx].copy()
X_test_without = X_without_history.loc[test_idx].copy()

# Set B: with previous campaign history
X_train_with = X_with_history.loc[train_idx].copy()
X_test_with = X_with_history.loc[test_idx].copy()

# RQ1: full feature set
X_train_rq1 = X_rq1.loc[train_idx].copy()
X_test_rq1 = X_rq1.loc[test_idx].copy()

# Target
y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

In [207]:
# Check train/test dimensions
print("RQ1 train:", X_train_rq1.shape)
print("RQ1 test:", X_test_rq1.shape)

print("\nSet A train:", X_train_without.shape)
print("Set A test:", X_test_without.shape)

print("\nSet B train:", X_train_with.shape)
print("Set B test:", X_test_with.shape)

print("\ny train:", y_train.shape)
print("y test:", y_test.shape)

RQ1 train: (32940, 20)
RQ1 test: (8236, 20)

Set A train: (32940, 7)
Set A test: (8236, 7)

Set B train: (32940, 11)
Set B test: (8236, 11)

y train: (32940,)
y test: (8236,)


In [208]:
# Check class proportions after splitting
print("Training target proportions:")
print(y_train.value_counts(normalize=True))

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True))

Training target proportions:
y
0    0.887341
1    0.112659
Name: proportion, dtype: float64

Test target proportions:
y
0    0.887324
1    0.112676
Name: proportion, dtype: float64


In [209]:
# Confirm all feature sets use the same rows
print(
    "RQ1 and Set A same training rows:",
    X_train_rq1.index.equals(X_train_without.index)
)

print(
    "RQ1 and Set B same training rows:",
    X_train_rq1.index.equals(X_train_with.index)
)

print(
    "RQ1 and Set A same test rows:",
    X_test_rq1.index.equals(X_test_without.index)
)

print(
    "RQ1 and Set B same test rows:",
    X_test_rq1.index.equals(X_test_with.index)
)

RQ1 and Set A same training rows: True
RQ1 and Set B same training rows: True
RQ1 and Set A same test rows: True
RQ1 and Set B same test rows: True


## 8. Preprocessing

Categorical features are one-hot encoded, while numerical features are standardised.

The `"unknown"` values are retained as valid categorical levels because removing these records would discard a substantial portion of the dataset.

All preprocessing transformations are fitted only on the training data and then applied to the test data to avoid data leakage.

In [210]:
# Set A categorical features
categorical_without = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan"
]

# Set A numerical features
numerical_without = [
    "age"
]

In [211]:
# Set B categorical features
categorical_with = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "poutcome"
]

# Set B numerical features
numerical_with = [
    "age",
    "previous",
    "pdays_clean"
]

# Binary indicator does not require scaling
binary_with = [
    "pdays_recorded"
]

In [212]:
# RQ1 categorical features
categorical_rq1 = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome"
]

# RQ1 numerical features
numerical_rq1 = [
    "age",
    "campaign",
    "previous",
    "pdays_clean",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
]

# RQ1 binary feature
binary_rq1 = [
    "pdays_recorded"
]

print(
    "RQ1 total raw features:",
    len(categorical_rq1) + len(numerical_rq1) + len(binary_rq1)
)

RQ1 total raw features: 20


In [213]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocessor for Set A
preprocessor_without = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_without
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_without
        )
    ]
)

In [214]:
# Preprocessor for Set B
preprocessor_with = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_with
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_with
        ),
        (
            "binary",
            "passthrough",
            binary_with
        )
    ]
)

In [215]:
# Preprocessor for RQ1
preprocessor_rq1 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_rq1
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_rq1
        ),
        (
            "binary",
            "passthrough",
            binary_rq1
        )
    ]
)

In [216]:
# Fit preprocessing on Set A training data
X_train_without_processed = preprocessor_without.fit_transform(
    X_train_without
)

# Apply the same preprocessing to Set A test data
X_test_without_processed = preprocessor_without.transform(
    X_test_without
)

In [217]:
# Fit preprocessing on Set B training data
X_train_with_processed = preprocessor_with.fit_transform(
    X_train_with
)

# Apply the same preprocessing to Set B test data
X_test_with_processed = preprocessor_with.transform(
    X_test_with
)

In [218]:
# Fit preprocessing on RQ1 training data
X_train_rq1_processed = preprocessor_rq1.fit_transform(
    X_train_rq1
)

# Apply the same preprocessing to RQ1 test data
X_test_rq1_processed = preprocessor_rq1.transform(
    X_test_rq1
)

In [219]:
# Check processed dimensions
print("RQ1 train:", X_train_rq1_processed.shape)
print("RQ1 test:", X_test_rq1_processed.shape)

print("\nSet A train:", X_train_without_processed.shape)
print("Set A test:", X_test_without_processed.shape)

print("\nSet B train:", X_train_with_processed.shape)
print("Set B test:", X_test_with_processed.shape)

RQ1 train: (32940, 63)
RQ1 test: (8236, 63)

Set A train: (32940, 34)
Set A test: (8236, 34)

Set B train: (32940, 40)
Set B test: (8236, 40)


In [220]:
# Get processed feature names
feature_names_rq1 = preprocessor_rq1.get_feature_names_out()
feature_names_without = preprocessor_without.get_feature_names_out()
feature_names_with = preprocessor_with.get_feature_names_out()

print("RQ1 feature count:", len(feature_names_rq1))
print(feature_names_rq1)

print("\nSet A feature count:", len(feature_names_without))
print(feature_names_without)

print("\nSet B feature count:", len(feature_names_with))
print(feature_names_with)

RQ1 feature count: 63
['categorical__job_admin.' 'categorical__job_blue-collar'
 'categorical__job_entrepreneur' 'categorical__job_housemaid'
 'categorical__job_management' 'categorical__job_retired'
 'categorical__job_self-employed' 'categorical__job_services'
 'categorical__job_student' 'categorical__job_technician'
 'categorical__job_unemployed' 'categorical__job_unknown'
 'categorical__marital_divorced' 'categorical__marital_married'
 'categorical__marital_single' 'categorical__marital_unknown'
 'categorical__education_basic.4y' 'categorical__education_basic.6y'
 'categorical__education_basic.9y' 'categorical__education_high.school'
 'categorical__education_illiterate'
 'categorical__education_professional.course'
 'categorical__education_university.degree'
 'categorical__education_unknown' 'categorical__default_no'
 'categorical__default_unknown' 'categorical__default_yes'
 'categorical__housing_no' 'categorical__housing_unknown'
 'categorical__housing_yes' 'categorical__loan_no'


In [221]:
# Convert processed RQ1 data to DataFrames
X_train_rq1_processed = pd.DataFrame(
    X_train_rq1_processed,
    columns=feature_names_rq1,
    index=X_train_rq1.index
)

X_test_rq1_processed = pd.DataFrame(
    X_test_rq1_processed,
    columns=feature_names_rq1,
    index=X_test_rq1.index
)

# Convert processed Set A to DataFrames
X_train_without_processed = pd.DataFrame(
    X_train_without_processed,
    columns=feature_names_without,
    index=X_train_without.index
)

X_test_without_processed = pd.DataFrame(
    X_test_without_processed,
    columns=feature_names_without,
    index=X_test_without.index
)

# Convert processed Set B to DataFrames
X_train_with_processed = pd.DataFrame(
    X_train_with_processed,
    columns=feature_names_with,
    index=X_train_with.index
)

X_test_with_processed = pd.DataFrame(
    X_test_with_processed,
    columns=feature_names_with,
    index=X_test_with.index
)

In [222]:
# Final preprocessing checks
print("RQ1 missing values:",
      X_train_rq1_processed.isnull().sum().sum())

print("Set A missing values:",
      X_train_without_processed.isnull().sum().sum())

print("Set B missing values:",
      X_train_with_processed.isnull().sum().sum())

print(
    "RQ1 train rows match target:",
    X_train_rq1_processed.index.equals(y_train.index)
)

print(
    "RQ1 test rows match target:",
    X_test_rq1_processed.index.equals(y_test.index)
)

print(
    "Set A and Set B same training rows:",
    X_train_without_processed.index.equals(
        X_train_with_processed.index
    )
)

print(
    "Set A and Set B same test rows:",
    X_test_without_processed.index.equals(
        X_test_with_processed.index
    )
)

RQ1 missing values: 0
Set A missing values: 0
Set B missing values: 0
RQ1 train rows match target: True
RQ1 test rows match target: True
Set A and Set B same training rows: True
Set A and Set B same test rows: True


## 9. RQ2 Experimental Setup

RQ2 evaluates the additional predictive value of previous campaign history by comparing two feature sets:

- **Set A:** Demographic and financial information only.
- **Set B:** Set A plus previous campaign history.

For a fair comparison, both feature sets use the same train/test observations, validation strategy, model configuration, and evaluation metrics.

In [223]:
# Summary of the two RQ2 feature sets
rq2_feature_sets = {
    "Set A - Without History": {
        "raw_features": features_without_history,
        "processed_feature_count": len(feature_names_without)
    },
    "Set B - With History": {
        "raw_features": features_with_history,
        "processed_feature_count": len(feature_names_with)
    }
}

for name, info in rq2_feature_sets.items():
    print(name)
    print("Raw features:", info["raw_features"])
    print("Processed feature count:", info["processed_feature_count"])
    print()

Set A - Without History
Raw features: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
Processed feature count: 34

Set B - With History
Raw features: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'previous', 'poutcome', 'pdays_recorded', 'pdays_clean']
Processed feature count: 40



In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Evaluate a binary classification model
def evaluate_model(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

In [225]:
# Store RQ2 model comparison results
rq2_results = pd.DataFrame(
    columns=[
        "Model",
        "Feature Set",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]
)

rq2_results

,Model,Feature Set,Accuracy,Precision,Recall,F1,ROC-AUC


## B1 —10. Evaluation Protocol and Baseline

All models are evaluated using the same data partitions and metrics. The existing
stratified 80% training / 20% test split is retained. Hyperparameters are selected
using five-fold stratified cross-validation within the training set. Five folds
balance computational cost and repeated validation, while stratification preserves
the class proportions in each fold.

| Component | Specification |
| --- | --- |
| Target | Subscription: `yes = 1` (positive), `no = 0` (negative) |
| Train/test split | Stratified 80% / 20%, random seed 42 |
| Validation | Five-fold stratified cross-validation, random seed 42 |
| Model selection | Highest mean validation F1 for the positive class |
| Additional metrics | Accuracy, positive-class Precision and Recall, ROC-AUC |
| Prediction rule | Default model predictions; no threshold tuning |
| Preprocessing | Encoding and scaling fitted within each training fold |
| Baseline | Zero-R majority-class classifier |

The same validation folds are used across models and feature sets. After
hyperparameter selection, each selected pipeline will be refitted on the full
training set and evaluated on the held-out test set. Test results will not be
used to select hyperparameters.

ROC-AUC is reported as a supplementary measure of ranking performance using
probabilities or decision scores. It is not used for model selection.
The random split assesses generalisation to held-out observations from the
historical dataset, rather than to future campaigns.

### 10.1 Evaluation Metrics

Subscriptions account for approximately 11.27% of observations. A classifier that
always predicts `no` can therefore achieve approximately 88.73% accuracy without
identifying any subscribers. Positive-class F1 is the primary metric because it
combines precision and recall with equal emphasis.

- **Precision:** the proportion of predicted subscribers who actually subscribe.
- **Recall:** the proportion of actual subscribers correctly identified.
- **F1:** the harmonic mean of positive-class precision and recall.

$$F_1 = \frac{2PR}{P+R} = \frac{2TP}{2TP+FP+FN}$$

Binary F1 is calculated with `pos_label=1`. Accuracy, precision, recall and ROC-AUC
provide complementary descriptions of model performance. Exact ties in mean
validation F1 will be resolved using a predefined candidate order favouring
simpler configurations.

When no positive predictions are made, precision is undefined and is reported
as 0 using `zero_division=0`. The same convention is applied to all models.


In [226]:
import sys
import numpy as np
import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, confusion_matrix

RANDOM_STATE = 42
N_SPLITS = 5
PRIMARY_METRIC = "f1"

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1": make_scorer(f1_score, pos_label=1, zero_division=0),
    "roc_auc": "roc_auc"
}

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.5
NumPy: 2.1.3
pandas: 2.2.3
scikit-learn: 1.6.1


In [227]:
# CV indices below are positional indices within the existing training data.
# Use .iloc (not .loc) if applying them manually.
for X_current in (X_train_rq1, X_train_without, X_train_with):
    assert X_current.index.equals(y_train.index)
assert X_train_rq1.index.intersection(X_test_rq1.index).empty

cv = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
cv_splits = list(cv.split(X_train_rq1, y_train))

fold_rows = []
for fold, (fit_pos, val_pos) in enumerate(cv_splits, start=1):
    fold_rows.append({
        "Fold": fold,
        "Training samples": len(fit_pos),
        "Validation samples": len(val_pos),
        "Training yes proportion": y_train.iloc[fit_pos].mean(),
        "Validation yes proportion": y_train.iloc[val_pos].mean()
    })

fold_summary = pd.DataFrame(fold_rows)
display(fold_summary)

,Fold,Training samples,Validation samples,Training yes proportion,Validation yes proportion
0,1,26352,6588,0.112667,0.112629
1,2,26352,6588,0.112667,0.112629
2,3,26352,6588,0.112667,0.112629
3,4,26352,6588,0.112667,0.112629
4,5,26352,6588,0.112629,0.112781


### 10.2 Preprocessing Within Cross-Validation

Each model pipeline receives the raw training features and an unfitted copy of
the corresponding preprocessor. One-hot encoding and standardisation are fitted
on each fold's training observations and then applied to its validation observations.
This prevents validation data from influencing preprocessing.

The matrices previously transformed using the full training set are not used
as inputs to cross-validation. Separate preprocessors are used for the RQ1
feature set and the two RQ2 feature sets.

In [228]:
def make_model_pipeline(preprocessor, model):
    return Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(model))
    ])

metric_names = {
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "roc_auc": "ROC-AUC"
}

def summarise_cv(cv_output):
    """Summarise validation-fold metrics, using population variance (ddof=0)."""
    rows = []
    for key, name in metric_names.items():
        scores = np.asarray(cv_output["test_" + key])
        rows.append({
            "Metric": name,
            "CV mean": scores.mean(),
            "CV variance": scores.var(ddof=0),
            "CV SD": scores.std(ddof=0)
        })
    return pd.DataFrame(rows).set_index("Metric")

### 10.3 Zero-R Baseline

Zero-R predicts the most frequent class in the training data for every observation.
It is implemented using `DummyClassifier(strategy="most_frequent")`. The majority
class is determined independently within each training fold and is `no = 0`
in all five folds.

The baseline does not use feature values and requires no encoding or scaling.
Its validation results are identical for RQ1 and both RQ2 feature sets because
they share the same labels and fold assignments. The baseline is therefore
evaluated once as a common reference.


In [229]:
baseline = DummyClassifier(strategy="most_frequent")

baseline_cv = cross_validate(
    baseline,
    X_train_rq1,
    y_train,
    cv=cv_splits,
    scoring=scoring,
    return_estimator=True,
    n_jobs=1,
    error_score="raise"
)

baseline_fold_results = pd.DataFrame({
    name: baseline_cv["test_" + key]
    for key, name in metric_names.items()
})
baseline_fold_results.index = np.arange(1, N_SPLITS + 1)
baseline_fold_results.index.name = "Fold"

baseline_summary = summarise_cv(baseline_cv)
display(baseline_fold_results)
display(baseline_summary)

,Accuracy,Precision,Recall,F1,ROC-AUC
Fold,,,,,
1,0.887371,0.0,0.0,0.0,0.5
2,0.887371,0.0,0.0,0.0,0.5
3,0.887371,0.0,0.0,0.0,0.5
4,0.887371,0.0,0.0,0.0,0.5
5,0.887219,0.0,0.0,0.0,0.5


,CV mean,CV variance,CV SD
Metric,,,
Accuracy,0.887341,3.686488e-09,0.000061
Precision,0.000000,0.000000e+00,0.000000
Recall,0.000000,0.000000e+00,0.000000
F1,0.000000,0.000000e+00,0.000000
ROC-AUC,0.500000,0.000000e+00,0.000000


In [230]:
# Reuse the fitted fold models; each observation is predicted by a model
# trained without that observation. This is a validation confusion matrix.
baseline_oof_pred = np.empty(len(y_train), dtype=int)
for fitted_baseline, (_, val_pos) in zip(baseline_cv["estimator"], cv_splits):
    baseline_oof_pred[val_pos] = fitted_baseline.predict(
        X_train_rq1.iloc[val_pos]
    )

baseline_oof_cm = pd.DataFrame(
    confusion_matrix(y_train, baseline_oof_pred, labels=[0, 1]),
    index=["Actual no (0)", "Actual yes (1)"],
    columns=["Predicted no (0)", "Predicted yes (1)"]
)
display(baseline_oof_cm)
print("Zero-R mean validation accuracy:",
      f"{baseline_summary.loc['Accuracy', 'CV mean']:.2%}")
print("Zero-R mean validation F1:",
      f"{baseline_summary.loc['F1', 'CV mean']:.4f}")

,Predicted no (0),Predicted yes (1)
Actual no (0),29229,0
Actual yes (1),3711,0


Zero-R mean validation accuracy: 88.73%
Zero-R mean validation F1: 0.0000


### 10.4 Baseline Results and Interpretation

Zero-R achieves **88.73% mean validation accuracy**, while positive-class
precision, recall and F1 are all **0**. Its constant probability score yields
**ROC-AUC = 0.5**. Across the out-of-fold predictions, all 29,229 non-subscribers
are classified correctly and all 3,711 subscribers are missed. This demonstrates
why accuracy alone is insufficient for this imbalanced task.

The confusion matrix aggregates validation predictions over the 32,940 training
observations; each prediction comes from a model trained without that observation.
These are cross-validation results, not held-out test results. Fold means,
variances and standard deviations describe variation across the five validation
folds; they are not confidence intervals.

Subsequent models will be compared with this baseline under the same evaluation
protocol. Exceeding Zero-R's F1 is a minimum reference rather than sufficient
evidence of practical usefulness. For RQ2, the two feature sets will use the same
model configuration, training rules and prediction rule to isolate the effect
of previous campaign history.

### 10.5 Reproducibility and Sources

Obtain `bank-additional-full.csv` from the
[UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing)
and place it at `data/bank-additional-full.csv` relative to this notebook.
Restart the kernel and run all cells in order. Software versions are recorded
above; the raw data is not embedded in this notebook.

Implementation references:
[scikit-learn DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)
and [scikit-learn cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html).

### RQ1: Logistic Regression

In [231]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

# Candidate C values
C_values = [0.01, 0.1, 1, 10, 100, 1000]

lr_rows = []

best_lr_C = None
best_lr_f1 = -1
best_lr_cv = None

for C in C_values:
    
    # Create Logistic Regression model
    lr = LogisticRegression(
        C=C,
        max_iter=2000
    )
    
    # Put preprocessing and model in the same pipeline
    lr_pipeline = make_model_pipeline(
        preprocessor_rq1,
        lr
    )
    
    # 5-fold cross-validation on training data
    cv_result = cross_validate(
        lr_pipeline,
        X_train_rq1,
        y_train,
        cv=cv_splits,
        scoring=scoring,
        n_jobs=1
    )
    
    # Calculate mean and SD of validation F1
    mean_f1 = cv_result["test_f1"].mean()
    sd_f1 = cv_result["test_f1"].std()
    
    lr_rows.append({
        "C": C,
        "Mean CV F1": mean_f1,
        "CV F1 SD": sd_f1
    })
    
    # Keep the C value with the highest mean CV F1
    if mean_f1 > best_lr_f1:
        best_lr_f1 = mean_f1
        best_lr_C = C
        best_lr_cv = cv_result


# Show tuning results
lr_tuning_results = pd.DataFrame(lr_rows)

display(lr_tuning_results)

print("Best C:", best_lr_C)
print("Best mean CV F1:", round(best_lr_f1, 4))

# Create final Logistic Regression using the best C
best_lr = LogisticRegression(
    C=best_lr_C,
    max_iter=2000
)

best_lr_pipeline = make_model_pipeline(
    preprocessor_rq1,
    best_lr
)

# Train on the entire training set
best_lr_pipeline.fit(
    X_train_rq1,
    y_train
)

# Predict the test set
lr_test_pred = best_lr_pipeline.predict(
    X_test_rq1
)

lr_test_prob = best_lr_pipeline.predict_proba(
    X_test_rq1
)[:, 1]

# Evaluate test performance
lr_test_metrics = evaluate_model(
    y_test,
    lr_test_pred,
    lr_test_prob
)

print("Logistic Regression test results:")
print(lr_test_metrics)

,C,Mean CV F1,CV F1 SD
0,0.01,0.310511,0.008168
1,0.10,0.336992,0.007883
2,1.00,0.342670,0.004887
3,10.00,0.343436,0.004704
4,100.00,0.344698,0.004927
5,1000.00,0.343789,0.004523


Best C: 100
Best mean CV F1: 0.3447
Logistic Regression test results:
{'Accuracy': 0.8983729966002914, 'Precision': 0.6491803278688525, 'Recall': 0.21336206896551724, 'F1': 0.32116788321167883, 'ROC-AUC': np.float64(0.8003312235764302)}


### RQ1: Descision Tree

In [232]:
from sklearn.tree import DecisionTreeClassifier

# Candidate maximum depths
max_depth_values = [1, 5, 10, 15, 20, None]

dt_rows = []

best_dt_depth = None
best_dt_f1 = -1
best_dt_cv = None

for depth in max_depth_values:
    
    # Create Decision Tree
    dt = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=depth,
        random_state=42
    )
    
    # Pipeline
    dt_pipeline = make_model_pipeline(
        preprocessor_rq1,
        dt
    )
    
    # Cross-validation
    cv_result = cross_validate(
        dt_pipeline,
        X_train_rq1,
        y_train,
        cv=cv_splits,
        scoring=scoring,
        n_jobs=1
    )
    
    mean_f1 = cv_result["test_f1"].mean()
    sd_f1 = cv_result["test_f1"].std()
    
    dt_rows.append({
        "max_depth": str(depth),
        "Mean CV F1": mean_f1,
        "CV F1 SD": sd_f1
    })
    
    # Keep best depth
    if mean_f1 > best_dt_f1:
        best_dt_f1 = mean_f1
        best_dt_depth = depth
        best_dt_cv = cv_result


# Show tuning results
dt_tuning_results = pd.DataFrame(dt_rows)

display(dt_tuning_results)

print("Best max_depth:", best_dt_depth)
print("Best mean CV F1:", round(best_dt_f1, 4))

# Create final Decision Tree
best_dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=best_dt_depth,
    random_state=42
)

best_dt_pipeline = make_model_pipeline(
    preprocessor_rq1,
    best_dt
)

# Train using whole training set
best_dt_pipeline.fit(
    X_train_rq1,
    y_train
)

# Predict test data
dt_test_pred = best_dt_pipeline.predict(
    X_test_rq1
)

dt_test_prob = best_dt_pipeline.predict_proba(
    X_test_rq1
)[:, 1]

# Test metrics
dt_test_metrics = evaluate_model(
    y_test,
    dt_test_pred,
    dt_test_prob
)

print("Decision Tree test results:")
print(dt_test_metrics)

,max_depth,Mean CV F1,CV F1 SD
0,1,0.000000,0.000000
1,5,0.364769,0.027528
2,10,0.371587,0.014494
3,15,0.373457,0.015579
4,20,0.345345,0.011008
5,None,0.319253,0.014126


Best max_depth: 15
Best mean CV F1: 0.3735
Decision Tree test results:
{'Accuracy': 0.8902379796017484, 'Precision': 0.5234375, 'Recall': 0.28879310344827586, 'F1': 0.37222222222222223, 'ROC-AUC': np.float64(0.6868212003142516)}


### RQ1: SVM

In [ ]:
from sklearn.svm import SVC

# Candidate C values
C_values_svm = [0.01, 0.1, 1]

svm_rows = []

best_svm_C = None
best_svm_f1 = -1
best_svm_cv = None

for C in C_values_svm:
    
    # Linear SVM, consistent with tutorial
    svm = SVC(
        kernel="linear",
        C=C
    )
    
    # Pipeline
    svm_pipeline = make_model_pipeline(
        preprocessor_rq1,
        svm
    )
    
    # Cross-validation
    cv_result = cross_validate(
        svm_pipeline,
        X_train_rq1,
        y_train,
        cv=cv_splits,
        scoring=scoring,
        n_jobs=-1
    )
    
    mean_f1 = cv_result["test_f1"].mean()
    sd_f1 = cv_result["test_f1"].std()
    
    svm_rows.append({
        "C": C,
        "Mean CV F1": mean_f1,
        "CV F1 SD": sd_f1
    })
    
    # Keep best C
    if mean_f1 > best_svm_f1:
        best_svm_f1 = mean_f1
        best_svm_C = C
        best_svm_cv = cv_result


# Show tuning results
svm_tuning_results = pd.DataFrame(svm_rows)

display(svm_tuning_results)

print("Best C:", best_svm_C)
print("Best mean CV F1:", round(best_svm_f1, 4))

# Create final SVM
best_svm = SVC(
    kernel="linear",
    C=best_svm_C
)

best_svm_pipeline = make_model_pipeline(
    preprocessor_rq1,
    best_svm
)

# Train on whole training set
best_svm_pipeline.fit(
    X_train_rq1,
    y_train
)

# Predict test classes
svm_test_pred = best_svm_pipeline.predict(
    X_test_rq1
)

# Decision scores for ROC-AUC
svm_test_score = best_svm_pipeline.decision_function(
    X_test_rq1
)

# Test metrics
svm_test_metrics = evaluate_model(
    y_test,
    svm_test_pred,
    svm_test_score
)

print("SVM test results:")
print(svm_test_metrics)

### B2 - Compare three model


In [56]:
rq1_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Best Hyperparameter": f"C = {best_lr_C}",
        "Mean CV F1": best_lr_f1,
        "Test Accuracy": lr_test_metrics["Accuracy"],
        "Test Precision": lr_test_metrics["Precision"],
        "Test Recall": lr_test_metrics["Recall"],
        "Test F1": lr_test_metrics["F1"],
        "Test ROC-AUC": lr_test_metrics["ROC-AUC"]
    },
    {
        "Model": "Decision Tree",
        "Best Hyperparameter": f"max_depth = {best_dt_depth}",
        "Mean CV F1": best_dt_f1,
        "Test Accuracy": dt_test_metrics["Accuracy"],
        "Test Precision": dt_test_metrics["Precision"],
        "Test Recall": dt_test_metrics["Recall"],
        "Test F1": dt_test_metrics["F1"],
        "Test ROC-AUC": dt_test_metrics["ROC-AUC"]
    },
    {
        "Model": "SVM",
        "Best Hyperparameter": f"C = {best_svm_C}",
        "Mean CV F1": best_svm_f1,
        "Test Accuracy": svm_test_metrics["Accuracy"],
        "Test Precision": svm_test_metrics["Precision"],
        "Test Recall": svm_test_metrics["Recall"],
        "Test F1": svm_test_metrics["F1"],
        "Test ROC-AUC": svm_test_metrics["ROC-AUC"]
    }
])

rq1_results = rq1_results.sort_values(
    by="Test F1",
    ascending=False
).reset_index(drop=True)

display(rq1_results.round(4))

NameError: name 'best_svm_C' is not defined

In [57]:
# Define the sets for RQ2
rq2_sets = {
    "Set A": {
        "X_train": X_train_without,
        "X_test": X_test_without,
        "preprocessor": preprocessor_without
    },
    "Set B": {
        "X_train": X_train_with,
        "X_test": X_test_with,
        "preprocessor": preprocessor_with
    }
}

### RQ2: Logistic Regression

In [65]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

C_values = [0.01, 0.1, 1, 10, 100, 1000]

lr_rq2_tuning = {}
lr_rq2_best = {}

for set_name, data in rq2_sets.items():

    rows = []

    best_C = None
    best_f1 = -1

    for C in C_values:

        lr = LogisticRegression(
            C=C,
            max_iter=2000
        )

        lr_pipeline = make_model_pipeline(
            data["preprocessor"],
            lr
        )

        cv_result = cross_validate(
            lr_pipeline,
            data["X_train"],
            y_train,
            cv=cv_splits,
            scoring=scoring,
            n_jobs=1,
            error_score="raise"
        )

        mean_f1 = cv_result["test_f1"].mean()
        sd_f1 = cv_result["test_f1"].std()

        rows.append({
            "C": C,
            "Mean CV F1": mean_f1,
            "CV F1 SD": sd_f1
        })

        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_C = C

    # Save tuning table
    lr_rq2_tuning[set_name] = pd.DataFrame(rows)

    # Save best parameter
    lr_rq2_best[set_name] = {
        "C": best_C,
        "Mean CV F1": best_f1
    }

print("Logistic Regression - Set A")
display(lr_rq2_tuning["Set A"])

print(
    "Best C:",
    lr_rq2_best["Set A"]["C"]
)

print(
    "Best CV F1:",
    round(lr_rq2_best["Set A"]["Mean CV F1"], 4)
)


print("\nLogistic Regression - Set B")
display(lr_rq2_tuning["Set B"])

print(
    "Best C:",
    lr_rq2_best["Set B"]["C"]
)

print(
    "Best CV F1:",
    round(lr_rq2_best["Set B"]["Mean CV F1"], 4)
)

lr_rq2_results = []

for set_name, data in rq2_sets.items():

    best_C = lr_rq2_best[set_name]["C"]

    final_lr = LogisticRegression(
        C=best_C,
        max_iter=2000
    )

    final_pipeline = make_model_pipeline(
        data["preprocessor"],
        final_lr
    )

    # Train on the whole training set
    final_pipeline.fit(
        data["X_train"],
        y_train
    )

    # Test predictions
    y_pred = final_pipeline.predict(
        data["X_test"]
    )

    y_prob = final_pipeline.predict_proba(
        data["X_test"]
    )[:, 1]

    # Calculate test metrics
    rq2_metrics = evaluate_model(
        y_test,
        y_pred,
        y_prob
    )

    # Save result for this feature set
    lr_rq2_results.append({
        "Model": "Logistic Regression",
        "Feature Set": set_name,
        "Best Parameter": f"C = {best_C}",
        "Mean CV F1": lr_rq2_best[set_name]["Mean CV F1"],
        **rq2_metrics
    })


# Convert results to table
lr_rq2_results_df = pd.DataFrame(lr_rq2_results)

display(lr_rq2_results_df.round(4))

Logistic Regression - Set A


,C,Mean CV F1,CV F1 SD
0,0.01,0.0,0.0
1,0.10,0.0,0.0
2,1.00,0.0,0.0
3,10.00,0.0,0.0
4,100.00,0.0,0.0
5,1000.00,0.0,0.0


Best C: 0.01
Best CV F1: 0.0

Logistic Regression - Set B


,C,Mean CV F1,CV F1 SD
0,0.01,0.259391,0.016598
1,0.10,0.300427,0.010263
2,1.00,0.304438,0.010155
3,10.00,0.304445,0.010928
4,100.00,0.304800,0.010598
5,1000.00,0.304800,0.010598


Best C: 100
Best CV F1: 0.3048


,Model,Feature Set,Best Parameter,Mean CV F1,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,Set A,C = 0.01,0.0000,0.8873,0.0000,0.0000,0.0000,0.6479
1,Logistic Regression,Set B,C = 100,0.3048,0.8956,0.6241,0.1843,0.2845,0.7014


### RQ2: Descision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate

max_depth_values = [1, 5, 10, 15, 20, None]

dt_rq2_tuning = {}
dt_rq2_best = {}

for set_name, data in rq2_sets.items():

    rows = []

    best_depth = None
    best_f1 = -1

    for depth in max_depth_values:

        dt = DecisionTreeClassifier(
            criterion="entropy",
            max_depth=depth,
            random_state=42
        )

        dt_pipeline = make_model_pipeline(
            data["preprocessor"],
            dt
        )

        cv_result = cross_validate(
            dt_pipeline,
            data["X_train"],
            y_train,
            cv=cv_splits,
            scoring=scoring,
            n_jobs=1,
            error_score="raise"
        )

        mean_f1 = cv_result["test_f1"].mean()
        sd_f1 = cv_result["test_f1"].std()

        rows.append({
            "max_depth": str(depth),
            "Mean CV F1": mean_f1,
            "CV F1 SD": sd_f1
        })

        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_depth = depth

    # Save tuning table
    dt_rq2_tuning[set_name] = pd.DataFrame(rows)

    # Save best parameter
    dt_rq2_best[set_name] = {
        "max_depth": best_depth,
        "Mean CV F1": best_f1
    }


# Display tuning results after both sets have finished
print("Decision Tree - Set A")
display(dt_rq2_tuning["Set A"])

print(
    "Best max_depth:",
    dt_rq2_best["Set A"]["max_depth"]
)

print(
    "Best CV F1:",
    round(dt_rq2_best["Set A"]["Mean CV F1"], 4)
)


print("\nDecision Tree - Set B")
display(dt_rq2_tuning["Set B"])

print(
    "Best max_depth:",
    dt_rq2_best["Set B"]["max_depth"]
)

print(
    "Best CV F1:",
    round(dt_rq2_best["Set B"]["Mean CV F1"], 4)
)

dt_rq2_results = []

for set_name, data in rq2_sets.items():

    best_depth = dt_rq2_best[set_name]["max_depth"]

    final_dt = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=best_depth,
        random_state=42
    )

    final_pipeline = make_model_pipeline(
        data["preprocessor"],
        final_dt
    )

    # Train on the whole training set
    final_pipeline.fit(
        data["X_train"],
        y_train
    )

    # Test predictions
    y_pred = final_pipeline.predict(
        data["X_test"]
    )

    y_prob = final_pipeline.predict_proba(
        data["X_test"]
    )[:, 1]

    # Calculate test metrics
    rq2_metrics = evaluate_model(
        y_test,
        y_pred,
        y_prob
    )

    # Save result for this feature set
    dt_rq2_results.append({
        "Model": "Decision Tree",
        "Feature Set": set_name,
        "Best Parameter": f"max_depth = {best_depth}",
        "Mean CV F1": dt_rq2_best[set_name]["Mean CV F1"],
        **rq2_metrics
    })


# Convert results to table
dt_rq2_results_df = pd.DataFrame(dt_rq2_results)

display(dt_rq2_results_df.round(4))

Decision Tree - Set A


,max_depth,Mean CV F1,CV F1 SD
0,1,0.000000,0.000000
1,2,0.014742,0.029484
2,3,0.014742,0.029484
3,5,0.047601,0.018928
4,10,0.071973,0.014726
5,15,0.125424,0.008136
6,None,0.140262,0.007906


Best max_depth: None
Best CV F1: 0.1403

Decision Tree - Set B


,max_depth,Mean CV F1,CV F1 SD
0,1,0.317154,0.009706
1,2,0.303220,0.010802
2,3,0.302364,0.011726
3,5,0.290023,0.012448
4,10,0.294960,0.014457
5,15,0.271566,0.012351
6,None,0.259794,0.008229


Best max_depth: 1
Best CV F1: 0.3172


,Model,Feature Set,Best Parameter,Mean CV F1,Accuracy,Precision,Recall,F1,ROC-AUC
0,Decision Tree,Set A,max_depth = None,0.1403,0.8628,0.2098,0.0787,0.1144,0.5829
1,Decision Tree,Set B,max_depth = 1,0.3172,0.8953,0.6071,0.2015,0.3026,0.5925


### RQ2: SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate

C_values_svm = [0.1, 1]

svm_rq2_tuning = {}
svm_rq2_best = {}

for set_name, data in rq2_sets.items():

    rows = []

    best_C = None
    best_f1 = -1

    for C in C_values_svm:

        svm = SVC(
            kernel="linear",
            C=C
        )

        svm_pipeline = make_model_pipeline(
            data["preprocessor"],
            svm
        )

        cv_result = cross_validate(
            svm_pipeline,
            data["X_train"],
            y_train,
            cv=cv_splits,
            scoring=scoring,
            n_jobs=-1,
            error_score="raise"
        )

        mean_f1 = cv_result["test_f1"].mean()
        sd_f1 = cv_result["test_f1"].std()

        rows.append({
            "C": C,
            "Mean CV F1": mean_f1,
            "CV F1 SD": sd_f1
        })

        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_C = C

    # Save tuning table
    svm_rq2_tuning[set_name] = pd.DataFrame(rows)

    # Save best parameter
    svm_rq2_best[set_name] = {
        "C": best_C,
        "Mean CV F1": best_f1
    }
    print("SVM - Set A")
display(svm_rq2_tuning["Set A"])

print(
    "Best C:",
    svm_rq2_best["Set A"]["C"]
)

print(
    "Best CV F1:",
    round(svm_rq2_best["Set A"]["Mean CV F1"], 4)
)


print("\nSVM - Set B")
display(svm_rq2_tuning["Set B"])

print(
    "Best C:",
    svm_rq2_best["Set B"]["C"]
)

print(
    "Best CV F1:",
    round(svm_rq2_best["Set B"]["Mean CV F1"], 4)
)

svm_rq2_results = []

for set_name, data in rq2_sets.items():

    best_C = svm_rq2_best[set_name]["C"]

    final_svm = SVC(
        kernel="linear",
        C=best_C
    )

    final_pipeline = make_model_pipeline(
        data["preprocessor"],
        final_svm
    )

    # Train on the whole training set
    final_pipeline.fit(
        data["X_train"],
        y_train
    )

    # Test predictions
    y_pred = final_pipeline.predict(
        data["X_test"]
    )

    # Use decision function for ROC-AUC
    y_score = final_pipeline.decision_function(
        data["X_test"]
    )

    # Calculate test metrics
    rq2_metrics = evaluate_model(
        y_test,
        y_pred,
        y_score
    )

    # Save result for this feature set
    svm_rq2_results.append({
        "Model": "SVM",
        "Feature Set": set_name,
        "Best Parameter": f"C = {best_C}",
        "Mean CV F1": svm_rq2_best[set_name]["Mean CV F1"],
        **rq2_metrics
    })

# Convert results to table
svm_rq2_results_df = pd.DataFrame(svm_rq2_results)

display(svm_rq2_results_df.round(4))

set_a = svm_rq2_results_df[
    svm_rq2_results_df["Feature Set"] == "Set A"
].iloc[0]

set_b = svm_rq2_results_df[
    svm_rq2_results_df["Feature Set"] == "Set B"
].iloc[0]

print("Change from Set A to Set B")

for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    change = set_b[metric] - set_a[metric]
    print(metric, ":", round(change, 4))